In [ ]:
#@title 1️⃣ 安装依赖 & 检查 FFmpeg
!pip install supabase requests tqdm pydub numpy scipy openai Pillow huggingface_hub>=0.20.0 -q

import shutil
import subprocess

if shutil.which("ffmpeg") and shutil.which("ffprobe"):
    print("✅ FFmpeg 已就绪（ffmpeg + ffprobe）")
else:
    subprocess.run(["apt-get", "-y", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "-y", "install", "-qq", "ffmpeg"], check=True)
    print("✅ 已安装 FFmpeg")

!pip install -q google-api-python-client google-auth-httplib2 google-auth-oauthlib

In [2]:
#@title 3️⃣ ⚙️ 配置参数（所有可调选项）

# ═══ Supabase 连接 ═══
SUPABASE_URL = ""  #@param {type:"string"}
SUPABASE_KEY = ""  #@param {type:"string"}

# ═══ 📺 YouTube 频道主配置 ═══
# 这里是项目主频道名；留给后面的上传、授权、项目标记默认继承
YOUTUBE_CHANNEL_NAME = ""  #@param {type:"string"} # 必须与 Supabase 凭证表 channel_name 匹配

# ═══ 任务与标记 ═══
# 一次最多处理的书籍数量（0=不限制）
MAX_PROCESS_COUNT = 10  #@param {type:"integer"}
# 当前项目标记名（防重复将以此为准，且不同项目不会互相覆盖；留空则自动使用 YOUTUBE_CHANNEL_NAME）
PROJECT_FLAG = ""  #@param {type:"string"}
if not str(PROJECT_FLAG).strip():
    PROJECT_FLAG = str(YOUTUBE_CHANNEL_NAME or "").strip()

# ═══ 输出设置 ═══
OUTPUT_ROOT = "/content/"  #@param {type:"string"}
# 要处理的分类名（Colab 下拉选项）
TARGET_CATEGORY = "文学小说"  #@param ["解读书", "文学小说", "国学经典", "政治军事", "名人传记", "历史博览", "社会科学", "科技科普", "成长励志", "职场精进", "商业财经", "管理艺术", "亲子教育", "青少年成长", "教育教学", "儿童文学", "少儿科普", "医学健康", "文艺风尚", "心理健康", "民俗文化", "哲学思想", "法律法规", "红色有声", "休闲娱乐", "评书相声", "广播剧"]

# ═══ 下载参数 ═══
DOWNLOAD_WORKERS = 4  #@param {type:"integer"}
REQUEST_DELAY = 0.3  #@param {type:"number"}
REQUEST_TIMEOUT = 30  #@param {type:"integer"}
MAX_RETRIES = 3  #@param {type:"integer"}
# 合并音频已存在时跳过下载
SKIP_EXISTING = True  #@param {type:"boolean"}
# 无视 status 标记，强制重新处理
FORCE_REPROCESS = False  #@param {type:"boolean"}
# Colab 最长运行 12 小时，建议预留缓冲避免跑到一半被系统强制结束
MAX_RUNTIME_HOURS = 11.5  #@param {type:"number"}
STOP_BUFFER_MINUTES = 20  #@param {type:"integer"}

# === 长音频分片 / 断点续跑 ===
# 预估总时长超过该值时，自动切成多个 YouTube 视频上传
LONG_AUDIO_SPLIT_TRIGGER_HOURS = 12.0  #@param {type:"number"}
# 单个分片的目标时长，建议略低于 12 小时，给元数据误差留余量
LONG_AUDIO_PART_TARGET_HOURS = 11.8  #@param {type:"number"}
# 长音频处理状态写入的 Supabase 表名
BOOK_STATE_TABLE = "book_processing_states"  #@param {type:"string"}
# 新一轮运行时，是否优先续跑上次未完成的长音频
PRIORITIZE_INTERRUPTED_BOOKS = True  #@param {type:"boolean"}

# ═══ 🔉 DeepFilter 降噪 ═══
ENABLE_DEEPFILTER = True  #@param {type:"boolean"}
segment_duration_minutes = 60  #@param {type:"integer"}
DEEPFILTER_WORKERS = 2  #@param {type:"integer"}

# ═══ 🖼️ YouTube AI 爆款封面生成 ═══
# 在所有音频完毕后利用书名和简介全自动结合大模型 Prompt 画图
ENABLE_COVER_GENERATION = True  #@param {type:"boolean"}
# MODELSCOPE_TOKEN 默认从 Supabase 云端读取；只有手动切到 local 才优先读本地
MODELSCOPE_TOKEN_SOURCE = "supabase"  #@param ["supabase", "local"]
# 云端运行时配置表（用于全局共享保存 MODELSCOPE_TOKEN / HF_DATASET_ZIP_URLS / BUCKET_IDS）
CLOUD_RUNTIME_SETTINGS_TABLE = "channel_runtime_settings"  #@param {type:"string"}
# Supabase 中存放 ModelScope Token 的表名（保留兼容；新版按全局共享方式使用）
MODELSCOPE_TOKEN_TABLE = "modelscope_tokens"  #@param {type:"string"}
# 本地 ModelScope Token（source=local 时直接使用；source=supabase 时仅在共享云端缺失时自动回填）
MODELSCOPE_TOKEN = ""  #@param {type:"string"}

# ═══ 📝 YouTube 爆款标题与描述文案生成 ═══
ENABLE_SEO_GENERATION = True  #@param {type:"boolean"}

# ═══ 📤 YouTube 全自动发帖入网分发节点设置 ═══
ENABLE_YOUTUBE_UPLOAD = True  #@param {type:"boolean"}
YOUTUBE_PRIVACY_STATUS = "schedule"  #@param ["private", "unlisted", "public", "schedule"]
YOUTUBE_SCHEDULE_AFTER_HOURS = 24  #@param {type:"integer"}
YOUTUBE_CATEGORY_ID = ""  #@param {type:"string"} # 留空=上传时不设置分类；22=人物与博客, 27=教育 等 (YouTube官方类别)
APPEND_TAGS_TO_TITLE = False  #@param {type:"boolean"} # 是否截取精选 tags 塞入标题末尾
APPEND_TAGS_TO_DESC = True  #@param {type:"boolean"} # 是否将全部 tags 铺在简介最底部

# ═══ 🎞️ 全套自动音视频融合封装设置 ═══
# 是否在完成海报图绘制配乐合并后组装为 MP4 (利用纯真帧速秒级渲染)
ENABLE_VIDEO_GENERATION = True  #@param {type:"boolean"}
# 自动生成的视频和封面画幅清晰度
VIDEO_RESOLUTION = "1080p"  #@param ["720p", "1080p", "1440p", "4k"]

# ═══ 🗑️ Hugging Face 音乐库设置 ═══
# 是否自动同步版权音乐到本地
DOWNLOAD_FROM_BUCKETS = True  #@param {type:"boolean"}
# 默认使用 datasets_zip_urls；若想继续使用旧的 Buckets 模式，再切回 buckets
HF_MUSIC_DOWNLOAD_METHOD = "datasets_zip_urls"  #@param ["datasets_zip_urls", "buckets"]
# HF_DATASET_ZIP_URLS 默认从 Supabase 共享云端读取；切到 local 才优先读本地
HF_DATASET_ZIP_URLS_SOURCE = "supabase"  #@param ["supabase", "local"]
# Hugging Face Datasets 的 ZIP 下载页地址，支持多个；可用英文逗号或换行分隔
HF_DATASET_ZIP_URLS = ""  #@param {type:"string"}
# BUCKET_IDS 默认从 Supabase 共享云端读取；切到 local 才优先读本地
BUCKET_IDS_SOURCE = "supabase"  #@param ["supabase", "local"]
# Bucket IDs（仅旧的 buckets 模式使用；多个库请用英文逗号分隔）
BUCKET_IDS = ""  #@param {type:"string"}
# 你的 HF Access Token（读取私有 Bucket / 私有 Dataset 时必需）
HF_TOKEN = ""  #@param {type:"string"}
# 本地存放目录 (同步完音乐存放在这里)
LOCAL_MUSIC_DIR = "/content/music"  #@param {type:"string"}

# ═══ BGM 混音 (基于 pydub 与频谱分析) ═══
# 是否在合并后自动混入背景音乐
ENABLE_BGM_MIX = True  #@param {type:"boolean"}
# 背景音乐目录（自动随机选择一首）
MUSIC_DIR = LOCAL_MUSIC_DIR  # 自动使用上方设置的本地音乐存放目录

# ─── 音量与滤波设置 ───
# 版权音乐相对于原始音频的音量偏移（dB）
VOLUME_OFFSET_DB = -25  #@param {type:"slider", min:-40, max:-10, step:1}
# 高通滤波截止频率（Hz），Content ID检测下限约140Hz
HIGHPASS_FREQ = 150  #@param {type:"slider", min:0, max:500, step:10}
# 淡入淡出时长（毫秒）用于音乐拼接点平衡过渡
FADE_DURATION_MS = 3000  #@param {type:"slider", min:500, max:10000, step:500}
# 版权音乐的最低绝对音量（dBFS），防止极低音量变静音
MIN_VOLUME_DB = -40  #@param {type:"slider", min:-60, max:-20, step:1}

# ─── 高级优化设置 ───
# 动态音量包络跟踪（BGM音量跟随原声音量起伏）
ENABLE_DYNAMIC_VOLUME = True  #@param {type:"boolean"}
# 频谱空隙嵌入（避开高能量人声频段填入BGM）
ENABLE_SPECTRAL_SHAPING = True  #@param {type:"boolean"}
# 立体声场偏移，微调可降低听觉干扰 (0=居中, 正=偏右, 负=偏左)
STEREO_OFFSET = 0.0  #@param {type:"slider", min:-1.0, max:1.0, step:0.1}



In [ ]:
#@title 3️⃣ 可选：创建项目所需的全部 Supabase 表（平时不用运行）
#@markdown 这个单元默认折叠。第一次初始化项目数据库时，直接复制下面 SQL 到 Supabase SQL Editor 执行即可。

state_table_name = str(BOOK_STATE_TABLE or "book_processing_states").strip() or "book_processing_states"
modelscope_token_table = str(MODELSCOPE_TOKEN_TABLE or "modelscope_tokens").strip() or "modelscope_tokens"
cloud_runtime_settings_table = str(CLOUD_RUNTIME_SETTINGS_TABLE or "channel_runtime_settings").strip() or "channel_runtime_settings"

CREATE_ALL_SUPABASE_TABLES_SQL = f"""
-- =========================================================
-- 1) books
-- 项目主数据表：存书名、分类、章节原始 JSON、处理状态与标签
-- 若你现有 books 表已经存在且还有额外字段，可保留额外字段不动
-- =========================================================
create table if not exists public.books (
  book_id text primary key,
  book_name text not null,
  category text not null default '',
  book_data jsonb not null default '{{}}'::jsonb,
  status text not null default '',
  tags text[] not null default '{{}}',
  created_at timestamptz not null default now(),
  updated_at timestamptz not null default now()
);

create index if not exists idx_books_category
  on public.books (category);

create index if not exists idx_books_tags_gin
  on public.books using gin (tags);


-- =========================================================
-- 2) youtube_credentials
-- 存放频道 OAuth 凭证，供自动上传与自动刷新 token 使用
-- =========================================================
create table if not exists public.youtube_credentials (
  channel_name text primary key,
  token_json jsonb not null,
  created_at timestamptz not null default now(),
  updated_at timestamptz not null default now()
);


-- =========================================================
-- 3) {modelscope_token_table}
-- 保留兼容的 ModelScope Token 表
-- 新版运行时统一使用固定共享作用域 __shared__，不再按频道区分
-- =========================================================
create table if not exists public.{modelscope_token_table} (
  channel_name text primary key,
  token_text text not null,
  created_at timestamptz not null default now(),
  updated_at timestamptz not null default now()
);

create index if not exists idx_{modelscope_token_table}_updated_at
  on public.{modelscope_token_table} (updated_at desc);


-- =========================================================
-- 4) {cloud_runtime_settings_table}
-- 全局共享保存可选云端运行配置，供默认读云端时使用
-- 虽然历史字段名仍叫 channel_name，但新版统一写入固定共享作用域 __shared__
-- 目前用于 MODELSCOPE_TOKEN / HF_DATASET_ZIP_URLS / BUCKET_IDS
-- =========================================================
create table if not exists public.{cloud_runtime_settings_table} (
  channel_name text not null,
  setting_key text not null,
  setting_value text not null default '',
  created_at timestamptz not null default now(),
  updated_at timestamptz not null default now(),
  constraint {cloud_runtime_settings_table}_pkey primary key (channel_name, setting_key)
);

create index if not exists idx_{cloud_runtime_settings_table}_updated_at
  on public.{cloud_runtime_settings_table} (updated_at desc);


-- =========================================================
-- 5) {state_table_name}
-- 长音频分片上传的断点续跑状态表
-- =========================================================
create table if not exists public.{state_table_name} (
  book_id text not null,
  project_flag text not null,
  book_name text,
  category text,
  pending_resume boolean not null default true,
  state_status text not null default 'in_progress',
  current_part_index integer,
  completed_part_count integer not null default 0,
  part_count integer not null default 1,
  updated_at timestamptz not null default now(),
  created_at timestamptz not null default now(),
  state_json jsonb not null default '{{}}'::jsonb,
  constraint {state_table_name}_pkey primary key (book_id, project_flag)
);

create index if not exists idx_{state_table_name}_pending_resume
  on public.{state_table_name} (project_flag, pending_resume, updated_at desc);

create index if not exists idx_{state_table_name}_category
  on public.{state_table_name} (category);
"""

print(CREATE_ALL_SUPABASE_TABLES_SQL)


In [ ]:
#@title 3️⃣ 可选：同步全局共享云端配置（默认关闭）
#@markdown 默认关闭。只有把下方开关打开后，才会把当前配置中的 `MODELSCOPE_TOKEN`、`HF_DATASET_ZIP_URLS`、`BUCKET_IDS` 写回 Supabase，供所有运行实例共用。
ENABLE_SYNC_CLOUD_RUNTIME_SETTINGS = False  #@param {type:"boolean"}

import datetime as dt_module

SHARED_CLOUD_SCOPE = "__shared__"


def _normalize_cloud_runtime_value(value):
    return str(value or "").replace("\r\n", "\n").replace("\r", "\n").strip()


def _upsert_or_delete_modelscope_token(client, token_value, table_name):
    normalized = _normalize_cloud_runtime_value(token_value)
    if normalized:
        client.table(table_name).upsert(
            {
                "channel_name": SHARED_CLOUD_SCOPE,
                "token_text": normalized,
                "updated_at": dt_module.datetime.now().isoformat(),
            }
        ).execute()
        return "saved"

    client.table(table_name).delete().eq("channel_name", SHARED_CLOUD_SCOPE).execute()
    return "deleted"


def _upsert_or_delete_runtime_setting(client, setting_key, setting_value, table_name):
    normalized = _normalize_cloud_runtime_value(setting_value)
    if normalized:
        client.table(table_name).upsert(
            {
                "channel_name": SHARED_CLOUD_SCOPE,
                "setting_key": setting_key,
                "setting_value": normalized,
                "updated_at": dt_module.datetime.now().isoformat(),
            }
        ).execute()
        return "saved"

    client.table(table_name).delete().eq("channel_name", SHARED_CLOUD_SCOPE).eq("setting_key", setting_key).execute()
    return "deleted"


if not ENABLE_SYNC_CLOUD_RUNTIME_SETTINGS:
    print("⏸️ 已保持关闭，本次不向 Supabase 同步云端运行配置。")
else:
    try:
        from supabase import create_client

        if not str(SUPABASE_URL).strip() or not str(SUPABASE_KEY).strip():
            raise ValueError("SUPABASE_URL / SUPABASE_KEY 不能为空")

        runtime_table_name = str(CLOUD_RUNTIME_SETTINGS_TABLE or "channel_runtime_settings").strip() or "channel_runtime_settings"
        modelscope_table_name = str(MODELSCOPE_TOKEN_TABLE or "modelscope_tokens").strip() or "modelscope_tokens"
        supabase_db = create_client(SUPABASE_URL, SUPABASE_KEY)

        modelscope_status = _upsert_or_delete_modelscope_token(
            supabase_db,
            MODELSCOPE_TOKEN,
            modelscope_table_name,
        )
        runtime_modelscope_status = _upsert_or_delete_runtime_setting(
            supabase_db,
            "MODELSCOPE_TOKEN",
            MODELSCOPE_TOKEN,
            runtime_table_name,
        )
        dataset_urls_status = _upsert_or_delete_runtime_setting(
            supabase_db,
            "HF_DATASET_ZIP_URLS",
            HF_DATASET_ZIP_URLS,
            runtime_table_name,
        )
        bucket_ids_status = _upsert_or_delete_runtime_setting(
            supabase_db,
            "BUCKET_IDS",
            BUCKET_IDS,
            runtime_table_name,
        )

        print(f"✅ 全局共享云端运行配置同步完成（作用域: {SHARED_CLOUD_SCOPE}）")
        print(f"   - MODELSCOPE_TOKEN（专用表 {modelscope_table_name}）: {modelscope_status}")
        print(f"   - MODELSCOPE_TOKEN（运行配置表 {runtime_table_name}）: {runtime_modelscope_status}")
        print(f"   - HF_DATASET_ZIP_URLS（运行配置表 {runtime_table_name}）: {dataset_urls_status}")
        print(f"   - BUCKET_IDS（运行配置表 {runtime_table_name}）: {bucket_ids_status}")
    except Exception as e:
        print("❌ 云端运行配置同步失败:", e)
        print("💡 如果提示表不存在，请先运行上一个“创建项目所需的全部 Supabase 表”单元，把 SQL 执行到 Supabase。")


In [ ]:
#@title 4️⃣ 🎈 首次初始化 YouTube 频道授权密钥 (含智能已授权探测)
#@markdown 运行此格前，请确保在下方填入想绑定的**频道代号**，并在目标路径上放好从 Google Cloud 申请下载的桌面端认证文件（如：`client_secret.json`）
BIND_CHANNEL_NAME = "" #@param {type:"string"} # 留空则自动使用上方配置中的 YOUTUBE_CHANNEL_NAME
if not str(BIND_CHANNEL_NAME).strip():
    BIND_CHANNEL_NAME = str(globals().get("YOUTUBE_CHANNEL_NAME", "") or "").strip()

CLIENT_SECRET_PATH = "/content/client_secret_470430300021-s5cu5uug1tgdi02rbf7jp9nlboi4pu02.apps.googleusercontent.com.json" #@param {type:"string"}
FORCE_REAUTH = False #@param {type:"boolean"} # 若已绑过想换一个Google账号对应此频道名，请勾选此强制项

import os
import json
from pathlib import Path
from urllib.parse import urlparse, parse_qs
from google_auth_oauthlib.flow import InstalledAppFlow


def resolve_client_secret_path(configured_path, search_root="/content"):
    try:
        root = Path(search_root)
        if root.exists():
            json_files = sorted(
                str(path)
                for path in root.rglob("*.json")
                if path.is_file() and "client_secret" in path.name.lower()
            )
            if json_files:
                chosen = json_files[0]
                print(f"🔎 已在 {search_root} 自动找到第一个文件名包含 client_secret 的 JSON 文件，将优先使用: {chosen}")
                return chosen
    except Exception as e:
        print(f"⚠️ 自动搜索 {search_root} 下 client_secret JSON 文件时出现问题，改为回退配置路径: {e}")

    fallback_path = str(configured_path or "").strip()
    if fallback_path:
        print(f"📍 未在 {search_root} 找到文件名包含 client_secret 的 JSON 文件，回退到配置路径: {fallback_path}")
    return fallback_path


def init_youtube_auth(channel, secret_path, force_update):
    try:
        from supabase import create_client
        if 'SUPABASE_URL' not in globals() or not SUPABASE_URL:
            print("❌ 请确保您先运行过上方第【3️⃣ ⚙️】参数配置单元格，以使系统记住您的数据库地址。")
            return

        supabase_db = create_client(SUPABASE_URL, SUPABASE_KEY)

        if not force_update:
            try:
                res = supabase_db.table("youtube_credentials").select("token_json").eq("channel_name", channel).execute()
                if getattr(res, "data", None) and len(res.data) > 0 and res.data[0].get("token_json"):
                    print(f"\n✅ 系统探测到：频道 [{channel}] 的无人值守通行证已经安全保存于 Supabase 中！")
                    print("💡 无需重复授权；后续主流程会直接复用这份凭证。")
                    print("   如果你想更换绑定账号，请勾选上面的 FORCE_REAUTH 后重新运行。")
                    return
            except Exception as inner_e:
                print("⚠️ 读取 Supabase 已有授权时遇到问题（如果是未建表，请先创建 youtube_credentials 表）:", inner_e)

    except Exception as e:
        print("❌ 配置环境探查异常:", e)
        return

    secret_path = resolve_client_secret_path(secret_path, search_root="/content")
    if not os.path.exists(secret_path):
        print(f"❌ 本地找不到 Desktop OAuth 客户端文件: {secret_path}")
        print("💡 请前往 Google Cloud Platform -> API & Services -> Credentials 下载 Desktop App JSON 并上传到上方路径。")
        return

    scopes = ['https://www.googleapis.com/auth/youtube']
    try:
        flow = InstalledAppFlow.from_client_secrets_file(secret_path, scopes)
        flow.redirect_uri = 'http://localhost'
        auth_url, _ = flow.authorization_url(prompt='consent')

        print("\n" + "=" * 80)
        print("🔗 请点击下方链接，用浏览器完成 Google 账号授权：")
        print(auth_url)
        print("=" * 80 + "\n")

        raw_input_str = input("👉 授权后，请把浏览器跳转到 localhost 的完整 URL 粘贴到这里：\n> ")
        parsed = urlparse(raw_input_str)
        code = parse_qs(parsed.query).get('code', [None])
        code = code[0] if code else None

        if not code:
            print("❌ 无法从回调 URL 中解析出授权 code，流程中断。")
            return

        print("⏳ 正在向 Google 兑换长期令牌...")
        flow.fetch_token(code=code)
        token_dict = json.loads(flow.credentials.to_json())

        data = {
            "channel_name": channel,
            "token_json": token_dict,
        }
        supabase_db.table("youtube_credentials").upsert(data).execute()
        print(f"🎉 授权成功！频道 [{channel}] 的自动上传凭证已写入 Supabase。")

    except Exception as e:
        print("❌ OAuth 授权流程失败:", e)


if not str(BIND_CHANNEL_NAME).strip():
    print("⚠️ BIND_CHANNEL_NAME 为空，且 YOUTUBE_CHANNEL_NAME 也为空，跳过授权初始化。")
else:
    init_youtube_auth(BIND_CHANNEL_NAME, CLIENT_SECRET_PATH, FORCE_REAUTH)


In [ ]:
#@title 5️⃣ 🌐 远端运行核心加载设置
REMOTE_PIPELINE_URL = "https://raw.githubusercontent.com/<your-account>/<your-repo>/main/audiobook_pipeline_runtime_core_v2.py"  #@param {type:"string"}
REMOTE_PIPELINE_LOCAL_PATH = "/content/_remote_pipeline/"  #@param {type:"string"}
REMOTE_PIPELINE_FORCE_REFRESH = True  #@param {type:"boolean"}


In [ ]:
#@title 6️⃣ 🚀 下载并运行远端核心
import importlib.util
import os
import time
from pathlib import Path
from urllib.parse import urlparse

import requests

RUNTIME_CONFIG_KEYS = ['SUPABASE_URL',
 'SUPABASE_KEY',
 'YOUTUBE_CHANNEL_NAME',
 'MAX_PROCESS_COUNT',
 'PROJECT_FLAG',
 'OUTPUT_ROOT',
 'TARGET_CATEGORY',
 'DOWNLOAD_WORKERS',
 'REQUEST_DELAY',
 'REQUEST_TIMEOUT',
 'MAX_RETRIES',
 'SKIP_EXISTING',
 'FORCE_REPROCESS',
 'MAX_RUNTIME_HOURS',
 'STOP_BUFFER_MINUTES',
 'LONG_AUDIO_SPLIT_TRIGGER_HOURS',
 'LONG_AUDIO_PART_TARGET_HOURS',
 'BOOK_STATE_TABLE',
 'PRIORITIZE_INTERRUPTED_BOOKS',
 'ENABLE_DEEPFILTER',
 'segment_duration_minutes',
 'DEEPFILTER_WORKERS',
 'ENABLE_COVER_GENERATION',
 'MODELSCOPE_TOKEN_SOURCE',
 'CLOUD_RUNTIME_SETTINGS_TABLE',
 'MODELSCOPE_TOKEN_TABLE',
 'MODELSCOPE_TOKEN',
 'ENABLE_SEO_GENERATION',
 'ENABLE_YOUTUBE_UPLOAD',
 'YOUTUBE_PRIVACY_STATUS',
 'YOUTUBE_SCHEDULE_AFTER_HOURS',
 'YOUTUBE_CATEGORY_ID',
 'APPEND_TAGS_TO_TITLE',
 'APPEND_TAGS_TO_DESC',
 'ENABLE_VIDEO_GENERATION',
 'VIDEO_RESOLUTION',
 'DOWNLOAD_FROM_BUCKETS',
 'HF_MUSIC_DOWNLOAD_METHOD',
 'HF_DATASET_ZIP_URLS_SOURCE',
 'HF_DATASET_ZIP_URLS',
 'BUCKET_IDS_SOURCE',
 'BUCKET_IDS',
 'HF_TOKEN',
 'LOCAL_MUSIC_DIR',
 'ENABLE_BGM_MIX',
 'MUSIC_DIR',
 'VOLUME_OFFSET_DB',
 'HIGHPASS_FREQ',
 'FADE_DURATION_MS',
 'MIN_VOLUME_DB',
 'ENABLE_DYNAMIC_VOLUME',
 'ENABLE_SPECTRAL_SHAPING',
 'STEREO_OFFSET']


def _ensure_remote_runtime_ready(remote_url, local_root, force_refresh=True):
    remote_url = str(remote_url or "").strip()
    if not remote_url:
        raise ValueError("REMOTE_PIPELINE_URL 不能为空。")
    if "<your-account>" in remote_url or "<your-repo>" in remote_url:
        raise ValueError("请先把 REMOTE_PIPELINE_URL 改成你自己的 GitHub Raw 地址。")

    local_root = Path(str(local_root or "/content/_remote_pipeline/").strip() or "/content/_remote_pipeline/")
    filename = os.path.basename(urlparse(remote_url).path) or "audiobook_pipeline_runtime_core_v2.py"
    if local_root.suffix == ".py":
        target_path = local_root
    else:
        target_path = local_root / filename

    target_path.parent.mkdir(parents=True, exist_ok=True)

    should_download = bool(force_refresh) or not target_path.exists() or target_path.stat().st_size == 0
    if should_download:
        download_url = remote_url
        if force_refresh:
            separator = "&" if "?" in remote_url else "?"
            download_url = f"{remote_url}{separator}t={int(time.time())}"

        response = requests.get(
            download_url,
            headers={"Cache-Control": "no-cache", "Pragma": "no-cache"},
            timeout=120,
        )
        response.raise_for_status()
        target_path.write_text(response.text, encoding="utf-8")
        print(f"✅ 已下载远端运行核心到: {target_path}")
    else:
        print(f"♻️ 复用本地已下载的运行核心: {target_path}")

    return target_path


def _collect_runtime_config_from_notebook_globals():
    runtime_config = {key: globals()[key] for key in RUNTIME_CONFIG_KEYS if key in globals()}
    runtime_config.update(
        {
            key: value
            for key, value in globals().items()
            if key.isupper() and not key.startswith("_")
        }
    )
    return runtime_config


remote_core_path = _ensure_remote_runtime_ready(
    REMOTE_PIPELINE_URL,
    REMOTE_PIPELINE_LOCAL_PATH,
    REMOTE_PIPELINE_FORCE_REFRESH,
)

module_name = f"audiobook_pipeline_runtime_core_v2_{int(time.time())}"
spec = importlib.util.spec_from_file_location(module_name, remote_core_path)
remote_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(remote_module)

runtime_config = _collect_runtime_config_from_notebook_globals()
runtime_result = remote_module.run_pipeline(runtime_config=runtime_config)
print("🎉 远端核心执行完成。")


## Loader Notebook 说明

这个新版 Notebook 只保留稳定配置和初始化操作，适合长期固定放在 Colab 中。

## 结构

1. 安装依赖和 FFmpeg
2. 配置所有运行参数
3. 可选：生成 Supabase 建表 SQL
4. 可选：同步全局共享云端配置
5. 首次初始化 YouTube 授权
6. 配置远端 GitHub Raw 运行核心地址
7. 下载并运行远端核心

## 日常使用

1. 运行安装依赖单元
2. 检查配置参数
3. 首次接入时执行 Supabase 建表和 YouTube 授权
4. 把 `REMOTE_PIPELINE_URL` 改成你 GitHub 上的 Raw 文件地址
5. 运行“下载并运行远端核心”

## 维护方式

- 平时只改 GitHub 上的 `audiobook_pipeline_runtime_core_v2.py`
- Colab 里的这个 loader notebook 一般不用重新上传
- 每次运行时会重新下载最新核心脚本
